In [ ]:
!nvidia-smi

Sun Aug 30 17:34:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile matrix_multiplication.cu

#include <iostream>
#include <cuda_runtime.h>

#define N 4

__global__ void matrixMultiply(float* A, float* B, float* C) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float sum = 0.0f;

        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}

int main() {
    int size = N * N * sizeof(float);

    float A[N * N] = {
        1, 2, 3, 4,
        5, 6, 7, 8,
        9, 10, 11, 12,
        13, 14, 15, 16
    };

    float B[N * N] = {
        1, 0, 0, 0,
        0, 1, 0, 0,
        0, 0, 1, 0,
        0, 0, 0, 1
    };

    float C[N * N];

    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(2, 2);

    dim3 numberOfBlocks(
        (N + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (N + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    matrixMultiply<<<numberOfBlocks, threadsPerBlock>>>(
        d_A,
        d_B,
        d_C
    );

    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, size, cudaMemcpyDeviceToHost);

    std::cout << "Result matrix C:" << std::endl;

    for (int row = 0; row < N; row++) {
        for (int col = 0; col < N; col++) {
            std::cout << C[row * N + col] << " ";
        }
        std::cout << std::endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix_multiplication.cu


In [ ]:
!nvcc -o matrix_multiplication matrix_multiplication.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matrix_multiplication

Result matrix C:
1 2 3 4 
5 6 7 8 
9 10 11 12 
13 14 15 16 


In [ ]:
!which nsys
!which ncu
!which nvprof

/usr/local/cuda/bin/ncu
/usr/local/cuda/bin/nvprof


In [ ]:
!ncu --set full --target-processes all ./matrix_multiplication

==PROF== Connected to process 2959 (/content/matrix_multiplication)
==PROF== Profiling "matrixMultiply" - 0: 0%....50%....100% - 31 passes
Result matrix C:
1 2 3 4 
5 6 7 8 
9 10 11 12 
13 14 15 16 
==PROF== Disconnected from process 2959
[2959] matrix_multiplication@127.0.0.1
  matrixMultiply(float *, float *, float *) (2, 2, 1)x(2, 2, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.82
    SM Frequency                    Mhz       574.95
    Elapsed Cycles                cycle        2,153
    Memory Throughput                 %         0.80
    DRAM Throughput                   %         0.35
    Duration                         us         3.74
    L1/TEX Cache Throughput           %         6.44
    L2 Cache Throughput               %       

In [ ]:
!ncu --set full --target-processes all ./matrix_multiplication > profiler_output.txt

In [ ]:
!head -100 profiler_output.txt

==PROF== Connected to process 3142 (/content/matrix_multiplication)
==PROF== Profiling "matrixMultiply" - 0: 0%....50%....100% - 31 passes
Result matrix C:
1 2 3 4 
5 6 7 8 
9 10 11 12 
13 14 15 16 
==PROF== Disconnected from process 3142
[3142] matrix_multiplication@127.0.0.1
  matrixMultiply(float *, float *, float *) (2, 2, 1)x(2, 2, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.79
    SM Frequency                    Mhz       574.89
    Elapsed Cycles                cycle        2,153
    Memory Throughput                 %         0.84
    DRAM Throughput                   %         0.35
    Duration                         us         3.74
    L1/TEX Cache Throughput           %         6.39
    L2 Cache Throughput               %       

In [ ]:
%%writefile cuda_benchmark.cu


#include <cuda_runtime.h>
#include <iostream>
#include <vector>
#include <chrono>
#include <iomanip>

#define TILE_SIZE 16

__global__ void matrixMultiply(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float sum = 0.0f;

        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}

void cpuMatrixMultiply(
    const std::vector<float>& A,
    const std::vector<float>& B,
    std::vector<float>& C,
    int N
) {
    for (int row = 0; row < N; row++) {
        for (int col = 0; col < N; col++) {
            float sum = 0.0f;

            for (int k = 0; k < N; k++) {
                sum += A[row * N + k] * B[k * N + col];
            }

            C[row * N + col] = sum;
        }
    }
}

int main() {
    std::vector<int> matrix_sizes = {256, 1024, 4096};

    std::cout << "Matrix Size,CPU Time (ms),GPU Kernel Time (ms),"
              << "Transfer Time (ms),GPU End-to-End Time (ms),Speedup"
              << std::endl;

    for (int N : matrix_sizes) {
        size_t number_of_elements = static_cast<size_t>(N) * N;
        size_t bytes = number_of_elements * sizeof(float);

        std::vector<float> A(number_of_elements, 1.0f);
        std::vector<float> B(number_of_elements, 1.0f);
        std::vector<float> C_cpu(number_of_elements, 0.0f);
        std::vector<float> C_gpu(number_of_elements, 0.0f);

        // CPU timing
        auto cpu_start = std::chrono::high_resolution_clock::now();

        cpuMatrixMultiply(A, B, C_cpu, N);

        auto cpu_end = std::chrono::high_resolution_clock::now();

        double cpu_time = std::chrono::duration<double, std::milli>(
            cpu_end - cpu_start
        ).count();

        float* d_A;
        float* d_B;
        float* d_C;

        cudaMalloc(&d_A, bytes);
        cudaMalloc(&d_B, bytes);
        cudaMalloc(&d_C, bytes);

        cudaEvent_t start_total, stop_total;
        cudaEvent_t start_kernel, stop_kernel;

        cudaEventCreate(&start_total);
        cudaEventCreate(&stop_total);
        cudaEventCreate(&start_kernel);
        cudaEventCreate(&stop_kernel);

        // Measure total GPU time: transfers plus kernel
        cudaEventRecord(start_total);

        cudaMemcpy(
            d_A, A.data(), bytes,
            cudaMemcpyHostToDevice
        );

        cudaMemcpy(
            d_B, B.data(), bytes,
            cudaMemcpyHostToDevice
        );

        dim3 threadsPerBlock(TILE_SIZE, TILE_SIZE);

        dim3 numberOfBlocks(
            (N + TILE_SIZE - 1) / TILE_SIZE,
            (N + TILE_SIZE - 1) / TILE_SIZE
        );

        cudaEventRecord(start_kernel);

        matrixMultiply<<<numberOfBlocks, threadsPerBlock>>>(
            d_A, d_B, d_C, N
        );

        cudaEventRecord(stop_kernel);

        cudaMemcpy(
            C_gpu.data(), d_C, bytes,
            cudaMemcpyDeviceToHost
        );

        cudaEventRecord(stop_total);
        cudaEventSynchronize(stop_total);

        float total_gpu_time = 0.0f;
        float kernel_time = 0.0f;

        cudaEventElapsedTime(
            &total_gpu_time,
            start_total,
            stop_total
        );

        cudaEventElapsedTime(
            &kernel_time,
            start_kernel,
            stop_kernel
        );

        float transfer_time = total_gpu_time - kernel_time;
        double speedup = cpu_time / total_gpu_time;

        std::cout << std::fixed << std::setprecision(3)
                  << N << ","
                  << cpu_time << ","
                  << kernel_time << ","
                  << transfer_time << ","
                  << total_gpu_time << ","
                  << speedup
                  << std::endl;

        cudaEventDestroy(start_total);
        cudaEventDestroy(stop_total);
        cudaEventDestroy(start_kernel);
        cudaEventDestroy(stop_kernel);

        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
    }

    return 0;
}

Overwriting cuda_benchmark.cu


In [ ]:
!nvcc -O3 cuda_benchmark.cu -o cuda_benchmark

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./cuda_benchmark

Matrix Size,CPU Time (ms),GPU Kernel Time (ms),Transfer Time (ms),GPU End-to-End Time (ms),Speedup
256,20.107,21.066,0.293,21.360,0.941
1024,3274.540,9.168,3.095,12.263,267.034
4096,817454.761,418.377,47.463,465.840,1754.797


In [ ]:
!ncu --set full --target-processes all ./cuda_benchmark > benchmark_profiler_output.txt

## CUDA Timing Results

| Matrix size | CPU (ms) | GPU kernel (ms) | H2D plus D2H (ms) | Speedup |
|---:|---:|---:|---:|---:|
| 256 | 20.107 | 21.066 | 0.293 | 0.941 |
| 1024 | 3274.540 | 9.168 | 3.095 | 267.034 |
| 4096 | 817454.761 | 418.377 | 47.463 | 1754.797 |